<a href="https://colab.research.google.com/github/rosyrosys/ai-soundscape/blob/main/notebooks/2_automatic_music_transcription.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# KSMPC 2026 여름학교 — MIR 기반 서양음악 연주 분석의 기초

## 2. 자동 음악 채보 (Automatic Music Transcription) 실습

**자동 음악 채보(AMT)**는 연주 음원에서 음표의 시작 시각, 종료 시각, 음높이, 세기 등을 추정해 MIDI와 같은 기호 음악 표현으로 바꾸는 기술입니다. 사람이 귀로 듣고 악보를 적는 과정을 컴퓨터가 수행한다고 생각하면 됩니다.

이 노트북에서는 서로 다른 악기와 목적에 맞게 설계된 세 모델을 살펴봅니다.

| 모델 | 주요 대상 | 특징 |
|---|---|---|
| [**TransKun**](https://github.com/yujia-yan/transkun) | 피아노 | 페달과 다성부를 포함한 피아노 연주 채보에 특화 |
| [**STRAdi**](https://github.com/seingreen/STRAdi) | 독주 바이올린 | 바이올린 음표를 추정하며 offline/online 구조를 제공 |
| [**Basic Pitch**](https://github.com/spotify/basic-pitch) | 다양한 악기 | 가볍게 실행할 수 있는 범용 다성 음악 채보 모델 |

모델마다 학습한 악기와 가정이 다르므로 같은 오디오라도 결과가 달라집니다. 아래에서는 원본 음원과 악보를 확인하고, 채보 결과를 piano roll로 시각화한 뒤 MIDI를 다시 합성해 들어봅니다.

## 0. 실습 환경 준비

처음 실행할 때 아래 setup cell을 한 번 실행합니다. 저장소와 실습 자료를 준비하고 TransKun, STRAdi, Basic Pitch 및 MIDI 합성에 필요한 package를 설치합니다.

Colab runtime을 새로 연결하면 설치 내용이 초기화되므로 다시 실행해야 합니다. GPU가 있으면 TransKun과 STRAdi가 자동으로 CUDA를 사용합니다.

In [ ]:
%cd /content
import os

if not os.path.isdir("/content/ksmpc2026/.git"):
    !git clone https://github.com/laurenceyoon/ksmpc2026.git /content/ksmpc2026
else:
    !git -C /content/ksmpc2026 pull --ff-only

%cd /content/ksmpc2026/notebooks

/content
Cloning into '/content/ksmpc2026'...


In [ ]:
# Install the required Linux programs and Python packages.
import subprocess
import sys

subprocess.run(
    ["apt-get", "update", "-qq"],
    check=True,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT,
)
subprocess.run(
    ["apt-get", "install", "-y", "-qq", "fluidsynth", "fluid-soundfont-gm"],
    check=True,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT,
)
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "transkun==2.0.1", "pretty_midi", "pyfluidsynth",
        "mir_eval", "resampy<0.4.3", "onnxruntime",
        "torchlibrosa", "mido", "soundfile",
    ],
    check=True,
)

# Use the ONNX model to avoid TensorFlow compatibility issues in Colab.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--no-deps", "basic-pitch==0.4.0"],
    check=True,
)

print("Package installation complete.")

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
RESOURCE_DIR = PROJECT_ROOT / "resources"
CHOPIN_AUDIOS = {
    "p09": RESOURCE_DIR / "Chopin_op10_no3_p09_short.wav",
    "p15": RESOURCE_DIR / "Chopin_op10_no3_p15_short.wav",
}
CHOPIN_SCORE = RESOURCE_DIR / "Chopin_op10_no3_p15_short.png"
BEETHOVEN_AUDIO = RESOURCE_DIR / "Beethoven_strqrt.wav"
BEETHOVEN_SCORE = RESOURCE_DIR / "Beethovon_String_Quartet_No.1_Op_18.png"
STRADI_AUDIO = RESOURCE_DIR / "Paganini_Caprice_No.24_Op.1_BomsoriKim.wav"
STRADI_SCORE = RESOURCE_DIR / "Paganini_Caprice_No.24_Op.1.png"
OUTPUT_DIR = PROJECT_ROOT / "results" / "automatic_music_transcription"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Import the libraries used throughout the notebook.
import numpy as np
import matplotlib.pyplot as plt
import pretty_midi
import IPython.display as ipd

print("Setup complete.")

# 1. TransKun을 이용한 피아노 자동 채보

[**TransKun**](https://github.com/yujia-yan/transkun)은 피아노 연주를 대상으로 학습된 자동 채보 모델입니다. 쇼팽 에튀드 Op. 10 No. 3의 같은 구간을 연주한 두 음원(`p09`, `p15`)을 입력으로 사용합니다.

악보를 보며 두 연주를 듣고 다음을 생각해 보세요.

- 한 시점에 몇 개의 음이 겹쳐 들리는가?
- sustain pedal로 길게 이어지는 음의 종료 시점을 모델이 어떻게 판단할까?
- 같은 악보라도 연주자에 따라 tempo와 dynamics가 어떻게 달라지는가?

두 연주는 같은 악보 구간이지만 길이와 timing이 다릅니다. 채보 결과 역시 연주마다 별도의 MIDI로 저장됩니다.

In [ ]:
# Display the score, then listen to the two excerpts used by TransKun.
display(ipd.Image(filename=str(CHOPIN_SCORE), width=900))

for excerpt, audio_path in CHOPIN_AUDIOS.items():
    print(f"Chopin Op. 10 No. 3 ({excerpt}) — {audio_path.name}")
    display(ipd.Audio(filename=str(audio_path)))

### 1.1 TransKun 실행

GPU가 연결되어 있으면 자동으로 CUDA를 사용하고, 그렇지 않으면 CPU로 실행합니다. 첫 실행은 model weight를 준비하므로 조금 더 오래 걸릴 수 있습니다. 각 WAV 파일에서 검출한 음표는 별도의 MIDI 파일로 저장됩니다.

In [ ]:
import torch

CHOPIN_MIDIS = {
    excerpt: OUTPUT_DIR / f"Chopin_op10_no3_{excerpt}_short_transkun.mid"
    for excerpt in CHOPIN_AUDIOS
}
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

for excerpt, audio_path in CHOPIN_AUDIOS.items():
    midi_path = CHOPIN_MIDIS[excerpt]
    print(f"Transcribing: {audio_path.name}")
    subprocess.run(
        ["transkun", "--device", device, str(audio_path), str(midi_path)],
        check=True,
    )
    print(f"Saved transcription: {midi_path}")

### 1.2 TransKun 결과 시각화

채보된 MIDI를 piano roll로 나타냅니다. 가로축은 시간(초), 세로축은 MIDI pitch이며 색이 밝을수록 velocity가 큽니다.

원본 악보와 비교하면서 빠진 음표, 잘못 추가된 음표, 지나치게 길거나 짧게 추정된 음표를 찾아보세요. 특히 pedal이 사용된 구간에서는 물리적인 건반 움직임과 실제로 들리는 음의 길이가 다를 수 있습니다.

In [ ]:
pm_chopin = {
    excerpt: pretty_midi.PrettyMIDI(str(midi_path))
    for excerpt, midi_path in CHOPIN_MIDIS.items()
}

fig, axes = plt.subplots(len(pm_chopin), 1, figsize=(12, 7), constrained_layout=True)
for ax, (excerpt, midi_data) in zip(axes, pm_chopin.items()):
    piano_roll = midi_data.get_piano_roll(fs=100)
    image = ax.imshow(
        piano_roll,
        origin="lower",
        aspect="auto",
        cmap="magma",
        extent=[0, midi_data.get_end_time(), 0, 127],
    )
    ax.set_title(f"TransKun: Chopin Op. 10 No. 3 ({excerpt})")
    ax.set_xlabel("Time (seconds)")
    ax.set_ylabel("MIDI pitch")
    ax.set_ylim(20, 110)
    fig.colorbar(image, ax=ax, label="Velocity")
plt.show()

### 1.3 생성된 MIDI 들어보기

채보 결과를 FluidSynth와 General MIDI piano soundfont로 합성합니다. 원본 연주와 번갈아 들으며 음높이뿐 아니라 onset, duration, velocity가 얼마나 자연스럽게 복원되었는지 확인합니다.

In [ ]:
SOUNDFONT = Path("/usr/share/sounds/sf2/FluidR3_GM.sf2")
if not SOUNDFONT.exists():
    soundfonts = list(Path("/usr/share/sounds").rglob("*.sf2"))
    if not soundfonts:
        raise FileNotFoundError("No SoundFont file was found.")
    SOUNDFONT = soundfonts[0]

for excerpt, midi_data in pm_chopin.items():
    print(f"TransKun synthesis ({excerpt})")
    chopin_synth = midi_data.fluidsynth(fs=44100, sf2_path=str(SOUNDFONT))
    display(ipd.Audio(chopin_synth, rate=44100))

# 2. STRAdi를 이용한 독주 바이올린 자동 채보

[**STRAdi**](https://github.com/seingreen/STRAdi)는 독주 바이올린 음원에서 note event를 추정하는 모델입니다. 예제로 바이올리니스트 김봄소리의 파가니니 Caprice No. 24, Op. 1 연주를 사용합니다. 먼저 악보와 원본 음원을 확인합니다.

이 실습에서는 **offline(non-causal) 모델**을 사용합니다. 곡 전체를 입력받은 뒤 과거와 미래의 문맥을 모두 참고할 수 있어, 실시간으로 처리하는 online(causal) 모델보다 채보에 유리하지만 연주가 끝나기 전에는 결과를 낼 수 없습니다.

### 2.1 STRAdi 준비

STRAdi source code와 offline checkpoint를 내려받아 프로젝트의 `.cache/STRAdi/`에 저장합니다. 이미 파일이 있으면 다시 내려받지 않습니다. 최신 PyTorch에서 공식 checkpoint를 읽을 수 있도록 loader의 호환성도 함께 처리합니다.

In [ ]:
import urllib.request

STRADI_DIR = PROJECT_ROOT / ".cache" / "STRAdi"
STRADI_CHECKPOINT = STRADI_DIR / "checkpoints" / "violin_transcription_offline.pth"
STRADI_CHECKPOINT_URL = (
    "https://github.com/seingreen/STRAdi/releases/download/v1.0.0/"
    "violin_transcription_offline.pth"
)

if not (STRADI_DIR / "transcribe.py").is_file():
    STRADI_DIR.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/seingreen/STRAdi.git", str(STRADI_DIR)],
        check=True,
    )

# PyTorch 2.6+ defaults to weights_only=True, but this trusted release
# checkpoint contains NumPy metadata in addition to model weights.
checkpoint_loader = STRADI_DIR / "violin_transcription" / "checkpoint.py"
loader_source = checkpoint_loader.read_text()
old_load_call = "torch.load(str(checkpoint_path), map_location=device)"
new_load_call = (
    "torch.load(str(checkpoint_path), map_location=device, weights_only=False)"
)
if old_load_call in loader_source:
    checkpoint_loader.write_text(loader_source.replace(old_load_call, new_load_call))

if not STRADI_CHECKPOINT.is_file():
    STRADI_CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
    print("Downloading the STRAdi offline checkpoint...")
    urllib.request.urlretrieve(STRADI_CHECKPOINT_URL, STRADI_CHECKPOINT)

display(ipd.Image(filename=str(STRADI_SCORE), width=900))
print(f"STRAdi input: {STRADI_AUDIO}")
display(ipd.Audio(filename=str(STRADI_AUDIO)))

### 2.2 피아노 모델 TransKun으로 바이올린 채보하기

같은 바이올린 음원을 TransKun에도 입력해 봅니다. TransKun은 피아노로 학습되었으므로 이는 모델의 본래 용도가 아닌 **out-of-domain 실험**입니다.

바이올린은 vibrato와 glissando로 음높이를 계속 바꿀 수 있지만 피아노는 그렇지 않습니다. 피아노 모델이 이러한 특성을 어떤 MIDI note로 근사하는지 관찰해 보세요. 합성할 때는 음표는 그대로 두고 General MIDI violin 음색을 사용합니다.

In [ ]:
PAGANINI_TRANSKUN_MIDI = OUTPUT_DIR / f"{STRADI_AUDIO.stem}_transkun.mid"
subprocess.run(
    ["transkun", "--device", device, str(STRADI_AUDIO), str(PAGANINI_TRANSKUN_MIDI)],
    check=True,
)

pm_paganini_transkun = pretty_midi.PrettyMIDI(str(PAGANINI_TRANSKUN_MIDI))
paganini_transkun_roll = pm_paganini_transkun.get_piano_roll(fs=100)

plt.figure(figsize=(12, 4))
plt.imshow(
    paganini_transkun_roll,
    origin="lower",
    aspect="auto",
    cmap="magma",
    extent=[0, pm_paganini_transkun.get_end_time(), 0, 127],
)
plt.title("TransKun transcription: Paganini Caprice No. 24")
plt.xlabel("Time (seconds)")
plt.ylabel("MIDI pitch")
plt.ylim(50, 115)
plt.colorbar(label="Velocity")
plt.show()

violin_program = pretty_midi.instrument_name_to_program("Violin")
for instrument in pm_paganini_transkun.instruments:
    if not instrument.is_drum:
        instrument.program = violin_program

paganini_transkun_synth = pm_paganini_transkun.fluidsynth(
    fs=44100, sf2_path=str(SOUNDFONT)
)
ipd.Audio(paganini_transkun_synth, rate=44100)

### 2.3 STRAdi offline 모델 실행

독주 바이올린에 특화된 offline 모델로 같은 음원을 채보합니다. 결과는 재생 가능한 **MIDI**와 각 음표의 onset, offset, MIDI pitch를 담은 **CSV**로 저장됩니다. 기본 onset/frame/offset threshold는 모두 0.5이며, threshold에 따라 검출되는 음표 수와 길이가 달라집니다.

In [ ]:
STRADI_OUTPUT_DIR = OUTPUT_DIR / "stradi_offline"
STRADI_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

stradi_process = subprocess.run(
    [
        sys.executable, str(STRADI_DIR / "transcribe.py"), str(STRADI_AUDIO),
        "--checkpoint", str(STRADI_CHECKPOINT),
        "--model", "offline",
        "--device", "cuda" if torch.cuda.is_available() else "cpu",
        "--output-dir", str(STRADI_OUTPUT_DIR),
    ],
    cwd=STRADI_DIR,
    capture_output=True,
    text=True,
)
print(stradi_process.stdout)
if stradi_process.returncode != 0:
    print(stradi_process.stderr)
    stradi_process.check_returncode()

# Match STRAdi's Path.with_suffix() output naming behavior.
stradi_output_stem = STRADI_OUTPUT_DIR / STRADI_AUDIO.stem
STRADI_MIDI = stradi_output_stem.with_suffix(".mid")
STRADI_CSV = stradi_output_stem.with_suffix(".csv")
print(f"Saved MIDI: {STRADI_MIDI}")
print(f"Saved note events: {STRADI_CSV}")

### 2.4 STRAdi 결과 확인

STRAdi가 검출한 음표를 piano roll로 표시하고 바이올린 음색으로 합성합니다. 앞의 TransKun 결과와 비교해 note onset, duration, pitch가 어떻게 달라졌는지 확인해 보세요.

MIDI는 연속적인 vibrato를 discrete pitch로 단순화하므로, 음표가 맞더라도 원본 연주의 미세한 음높이 변화와 표현은 완전히 재현되지 않습니다.

In [ ]:
pm_stradi = pretty_midi.PrettyMIDI(str(STRADI_MIDI))
stradi_roll = pm_stradi.get_piano_roll(fs=100)

plt.figure(figsize=(12, 4))
plt.imshow(
    stradi_roll,
    origin="lower",
    aspect="auto",
    cmap="magma",
    extent=[0, pm_stradi.get_end_time(), 0, 127],
)
plt.title("STRAdi offline transcription: Paganini Caprice No. 24")
plt.xlabel("Time (seconds)")
plt.ylabel("MIDI pitch")
plt.ylim(50, 115)
plt.colorbar(label="Velocity")
plt.show()

stradi_synth = pm_stradi.fluidsynth(fs=44100, sf2_path=str(SOUNDFONT))
ipd.Audio(stradi_synth, rate=44100)

# 3. Basic Pitch를 이용한 다중 악기 자동 채보

[**Basic Pitch**](https://github.com/spotify/basic-pitch)는 Spotify가 공개한 가벼운 범용 자동 채보 모델입니다. 다양한 악기의 polyphonic audio를 처리하도록 설계되었으며, 여기서는 베토벤 현악 4중주를 입력합니다.

현악 4중주에서는 활로 시작하는 음의 onset이 불분명하고, vibrato와 portamento로 pitch가 계속 움직이며, 네 악기의 음역이 서로 겹칩니다. 같은 pitch를 여러 악기가 함께 연주해도 하나의 MIDI note로 합쳐질 수 있습니다.

In [ ]:
# Display the Beethoven score and listen to the input audio.
display(ipd.Image(filename=str(BEETHOVEN_SCORE), width=1000))
display(ipd.Audio(filename=str(BEETHOVEN_AUDIO)))

### 3.1 Basic Pitch 실행

Colab 환경과의 호환성을 위해 ONNX 모델을 사용합니다. 추론 결과로 frame-level model output, MIDI 객체, note event 목록을 얻습니다. Basic Pitch는 pitch bend도 추정해 현악기의 미세한 음높이 변화를 일부 보존합니다.

In [ ]:
from basic_pitch.inference import predict
from basic_pitch import ICASSP_2022_MODEL_PATH

print(f"Model: {ICASSP_2022_MODEL_PATH}")
model_output, midi_data, note_events = predict(
    str(BEETHOVEN_AUDIO),
    model_or_model_path=ICASSP_2022_MODEL_PATH,
)

BEETHOVEN_MIDI = OUTPUT_DIR / "beethoven_basic_pitch.mid"
midi_data.write(str(BEETHOVEN_MIDI))
print(f"Detected notes: {len(note_events)}")
print(f"Saved MIDI: {BEETHOVEN_MIDI}")

### 3.2 추정된 음높이 분포

시간에 따른 pitch 후보의 확률을 표시합니다. 밝은 ridge가 모델이 감지한 음높이이며, vibrato가 있는 구간에서는 위아래로 흔들립니다. 여러 악기가 동시에 연주하면 여러 ridge가 겹쳐 나타납니다.

In [ ]:
plt.figure(figsize=(12, 5))
plt.imshow(
    model_output["contour"].T,
    aspect="auto",
    origin="lower",
    cmap="magma",
)
plt.title("Basic Pitch: pitch contour")
plt.xlabel("Frame")
plt.ylabel("Pitch bin")
plt.colorbar(label="Confidence")
plt.show()

### 3.3 MIDI piano roll

연속적인 pitch 확률에서 note onset과 offset을 결정해 MIDI note로 바꾼 결과입니다. 원본 악보와 비교해 어떤 성부가 잘 검출되고 어떤 음표가 합쳐지거나 빠졌는지 살펴봅니다.

In [ ]:
beethoven_roll = midi_data.get_piano_roll(fs=100)

plt.figure(figsize=(12, 4))
plt.imshow(
    beethoven_roll,
    origin="lower",
    aspect="auto",
    cmap="viridis",
    extent=[0, midi_data.get_end_time(), 0, 127],
)
plt.title("Basic Pitch transcription: Beethoven String Quartet")
plt.xlabel("Time (seconds)")
plt.ylabel("MIDI pitch")
plt.ylim(20, 110)
plt.colorbar(label="Velocity")
plt.show()

### 3.4 MIDI note와 pitch bend

파란 선은 quantized MIDI note, 빨간 선은 각 음표 안에서 추정한 연속적인 pitch 변화를 나타냅니다. 빨간 선의 흔들림으로 vibrato와 음정 이동이 어느 정도 포착되었는지 확인할 수 있습니다.

In [ ]:
plt.figure(figsize=(12, 4))

for start, end, pitch, velocity, pitch_bend in note_events:
    plt.plot([start, end], [pitch, pitch], color="royalblue", linewidth=2)
    if pitch_bend:
        times = np.linspace(start, end, len(pitch_bend))
        plt.plot(
            times,
            pitch + np.asarray(pitch_bend),
            color="crimson",
            alpha=0.55,
        )

plt.title("MIDI notes (blue) and pitch bends (red)")
plt.xlabel("Time (seconds)")
plt.ylabel("MIDI pitch")
plt.grid(alpha=0.2)
plt.show()

In [ ]:
# List and optionally download the generated files.
from google.colab import files

print("Generated files:")
for path in sorted(OUTPUT_DIR.rglob("*.mid")):
    print(" -", path.relative_to(OUTPUT_DIR))

# Uncomment these lines to download the files from Colab.
# for path in CHOPIN_MIDIS.values():
#     files.download(str(path))
# files.download(str(BEETHOVEN_MIDI))
# files.download(str(STRADI_MIDI))
# files.download(str(STRADI_CSV))

# References

[1] [TransKun GitHub](https://github.com/yujia-yan/transkun)

[2] [STRAdi GitHub](https://github.com/seingreen/STRAdi)

[3] [Basic Pitch GitHub](https://github.com/spotify/basic-pitch)

[4] [GCT731-2026 GitHub](https://github.com/juhannam/gct731-2026)

[5] Y. Yan and Z. Duan, Scoring Time Intervals Using Non-Hierarchical Transformer for Automatic Piano Transcription, ISMIR 2024.

[6] Y. Yan, F. Cwitkowitz, and Z. Duan, Skipping the Frame-Level: Event-Based Piano Transcription With Neural Semi-CRFs, NeurIPS 2021.

[7] S. Lee, J. Park, and J. Nam, STRAdi: Real-time Violin Transcription for Score Following, ISMIR 2026 (Accepted).

[8] R. M. Bittner, J. Bosch, D. Rubinstein, G. Meseguer-Brocal, and S. Ewert,
 A Lightweight Instrument-Agnostic Model for Polyphonic Note Transcription and Multipitch Estimation, ICASSP 2022.